# Attribute Lookup, Properties, and Descriptors
Instead of treating **Attribute Lookup, Properties, and Descriptors** as a small isolated topic, the goal here is to treat it as part of Python's larger runtime story: objects are created, names are bound, frames execute bytecode, and the interpreter keeps following rules whether or not the source code looks simple from the outside.

If this topic ever felt a little too neat or too magical when explained quickly, that is exactly what this rewrite is trying to fix. We are going to keep asking two grounding questions: what changed in memory, and what instructions is the interpreter actually stepping through under the surface?

The point is not to drown in internals. The point is to make the language feel explainable. Once that happens, even advanced behavior starts feeling much less mysterious.


When people struggle with **Attribute Lookup, Properties, and Descriptors**, the struggle is often not about syntax. It is usually about trying to reason from surface appearance alone. Python rewards a deeper habit: do not ask only what the code *looks like*; ask what objects exist, what names or attributes point to them, what bytecode-level actions are likely happening, and which runtime protocol is being used.

That habit is what turns memorized rules into real understanding. It also makes interviews easier, because you stop giving slogan answers and start giving mechanism-based answers.


- Understand attribute lookup more deeply
- See properties as controlled attribute access
- Demystify descriptors
- Connect this topic to methods and framework internals


An object often has its own instance dictionary, but the class may also define descriptors that change how reads and writes behave. That means attribute access is not merely “look in a dict and stop”.


In [1]:
class Demo:
    def __init__(self, x):
        self._x = x

    @property
    def x(self):
        return self._x

d = Demo(5)
print(d.__dict__)
print(Demo.__dict__["x"])
print(d.x)


{'_x': 5}
5


Disassembly of a property-using method reminds us that attribute access is a real interpreter operation. Python performs load-attribute steps that may trigger descriptor logic.


In [2]:
import dis

class Demo:
    def __init__(self, x):
        self._x = x
    @property
    def x(self):
        return self._x

def read_value(obj):
    return obj.x

dis.dis(read_value)


 10           0 RESUME                   0

 11           2 LOAD_FAST                0 (obj)
              4 LOAD_ATTR                0 (x)
             14 RETURN_VALUE


Instance data, class data, and descriptors all matter.

They let callers use attribute syntax while the class keeps control.

Properties, methods, and many framework features are built on this idea.

It turns surprising behavior into expected behavior.


This keeps attribute-style use while still enforcing rules.


In [3]:
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = value

t = Temperature(25)
t.celsius = 30
print(t.celsius)


30


This custom object controls how the attribute behaves when read or written.


In [4]:
class PositiveNumber:
    def __set_name__(self, owner, name):
        self.private_name = "_" + name
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.private_name)
    def __set__(self, instance, value):
        if value <= 0:
            raise ValueError("must be positive")
        setattr(instance, self.private_name, value)

class Product:
    price = PositiveNumber()
    def __init__(self, price):
        self.price = price

p = Product(10)
print(p.price)


10


Methods becoming bound is part of the same broad story.


In [5]:
class Greeter:
    def hello(self):
        return "hi"

g = Greeter()
print(Greeter.hello)
print(g.hello)


<function Greeter.hello at 0x000001E78DC0B6A0>
<bound method Greeter.hello of <__main__.Greeter object at 0x000001E78DC52410>>


This is not only a classroom topic. **Attribute Lookup, Properties, and Descriptors** usually appears in real projects as a debugging problem, a design choice, a performance tradeoff, or a readability decision. In other words, this topic matters most when the codebase gets large enough that vague intuition stops being reliable.

That is why the low-level view is useful. It gives you something firmer than guesswork when code starts behaving in surprising ways.


- Thinking properties are only fancy getters and setters with no runtime model behind them
- Jumping into descriptors before understanding attribute lookup basics
- Using hidden heavy work behind properties without making it clear


- What is a property?
- What is a descriptor?
- Why do methods become bound on instances?


- Attribute access is a real lookup process.
- Properties are friendly descriptor-based tools.
- Descriptors explain a surprising amount of Python.


If this notebook did its job, **Attribute Lookup, Properties, and Descriptors** should now feel less like a bag of special rules and more like a natural consequence of Python's runtime model. That is the standard we want: not just recall, but a calm explanation of what the interpreter is seeing and why the behavior follows from that.


A better way to understand Attribute Lookup, Properties, and Descriptors is to keep moving back and forth between three views of the same code.

The first view is the human view. This is what the code is trying to say. The second view is the object view. In that view, we ask which Python objects exist, which names refer to them, and which objects are being created, reused, mutated, or discarded. The third view is the interpreter view. In that view, we ask what protocol is being used, what bytecode is being executed, and which runtime rules are controlling the result.

Many learners stop after the first view. That is enough to copy code, but it is usually not enough to explain surprising behavior. Deep understanding starts when these three views become one mental movement instead of three unrelated topics.


There is also a fourth habit that helps a lot: failure-based thinking.

For Attribute Lookup, Properties, and Descriptors, ask yourself what normally goes wrong when the idea is only half-understood. Does someone confuse identity with equality? Do they expect copying where only rebinding happened? Do they expect a fresh iterator when they really have an exhausted one? Do they trust a default value that is secretly shared? Do they think a class attribute is per-instance state? Once you can predict the common failure modes before they happen, your understanding becomes much more stable.


In [6]:
import dis
import inspect

def probe_runtime(value):
    local_copy = value
    frame = inspect.currentframe()
    print('locals now:', frame.f_locals)
    return local_copy

print(probe_runtime('demo'))
print('\nbytecode for probe_runtime:')
dis.dis(probe_runtime)


locals now: {'value': 'demo', 'local_copy': 'demo', 'frame': <frame at 0x000001E78DA87F40, file 'C:\\Users\\VIHAAN\\AppData\\Local\\Temp\\ipykernel_30120\\939914315.py', line 7, code probe_runtime>}
demo

bytecode for probe_runtime:
  4           0 RESUME                   0

  5           2 LOAD_FAST                0 (value)
              4 STORE_FAST               1 (local_copy)

  6           6 LOAD_GLOBAL              0 (inspect)
             18 LOAD_METHOD              1 (currentframe)
             40 PRECALL                  0
             44 CALL                     0
             54 STORE_FAST               2 (frame)

  7          56 LOAD_GLOBAL              5 (NULL + print)
             68 LOAD_CONST               1 ('locals now:')
             70 LOAD_FAST                2 (frame)
             72 LOAD_ATTR                3 (f_locals)
             82 PRECALL                  2
             86 CALL                     2
             96 POP_TOP

  8          98 LOAD_FAST        

The deeper object-model gain here is to inspect instance namespaces, class namespaces, method resolution order, and descriptor objects directly. That shifts the topic from ?class syntax? to ?runtime lookup process?, which is where the real explanatory power lives.


In [7]:
class Base:
    kind = 'base'
    def hello(self):
        return 'hello'

class Child(Base):
    pass

obj = Child()
print('instance dict:', obj.__dict__)
print('class dict sample:', list(Child.__dict__.keys())[:8])
print('mro:', Child.__mro__)
print('bound method:', obj.hello)


instance dict: {}
class dict sample: ['__module__', '__doc__']
mro: (<class '__main__.Child'>, <class '__main__.Base'>, <class 'object'>)
bound method: <bound method Base.hello of <__main__.Child object at 0x000001E78DC557D0>>


This chapter is where many invisible runtime rules become visible. Attribute access can look simple from the outside, but under the surface it is one of the richest lookup processes in the language. Instance dictionaries, class dictionaries, descriptors, and method binding all meet here. That is why this topic is so important for deeper understanding of frameworks and advanced class behavior.


## Final Takeaway

The deepest way to revise **Attribute Lookup, Properties, and Descriptors** is to stop treating it as a collection of syntax facts and instead treat it as a runtime story. Ask what objects are involved, what names or attributes point to those objects, what frame is active, and what protocol or bytecode path Python is following. If you can explain this topic at those four levels, then you understand much more than the visible syntax.

In interview terms, the strongest answer is usually not the shortest definition. It is a calm explanation of what Python is doing under the surface and why the behavior follows from that model. For revision, come back to this notebook and ask yourself: could I explain this entire chapter with one small code example, one memory-level explanation, and one interpreter-level explanation? If yes, the topic is becoming yours.
